# Module 5: LangChain
# Topic 50: Error Handling & Retry

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Production Focus)
>
> **Interview Frequency:** High
>
> **Production Importance:** Critical
>
> **Prerequisites:**
> - LCEL ✅
> - Agents ✅
> - RAG ✅
> - Callbacks ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- Why Error Handling is important
- Common failures in LLM applications
- Retry strategies
- Timeout handling
- Fallback models
- Exception handling
- Production best practices
- Interview questions

---

# 1. Why Do We Need Error Handling?

Unlike traditional software, an LLM application depends on many external services.

Example:

```text
User

↓

Application

↓

LLM API

↓

Embedding API

↓

Vector DB

↓

External APIs
```

Any one of these can fail.

Without proper error handling:

```
Application Crashes
```

---

# Interview Definition ⭐⭐⭐⭐⭐

> **Error Handling is the process of detecting, managing, recovering from, and reporting failures during AI application execution without disrupting the user experience.**

---

# 2. Common Failures in LangChain Applications

| Component | Possible Error |
|------------|----------------|
| LLM | API unavailable |
| Embeddings | Rate limit |
| Vector DB | Connection timeout |
| Retriever | No documents found |
| Tool | Invalid API response |
| Output Parser | Invalid JSON |
| Agent | Infinite reasoning loop |

---

# 3. Error Handling Architecture

```text
User

↓

Application

↓

LLM

↓

Success?

│

├── Yes

│      ↓

│   Continue

│

└── No

       ↓

 Retry?

 │

 ├── Yes

 │

 ▼

 Retry

 │

 ▼

 Success?

 │

 ├── Yes

 │

 ▼

 Continue

 │

 └── No

       ↓

Fallback

↓

Return Safe Response
```

---

# 4. Types of Errors

### Network Errors

```
Connection Timeout

DNS Failure

SSL Error
```

---

### API Errors

```
401 Unauthorized

403 Forbidden

429 Too Many Requests

500 Internal Server Error
```

---

### Application Errors

```
Invalid Prompt

Missing Variables

Parser Failure

Tool Failure
```

---

# 5. Basic Exception Handling

```python
try:
    response = llm.invoke(
        "Explain RAG."
    )
    print(response.content)

except Exception as e:
    print(f"LLM Error: {e}")
```

This prevents the application from crashing.

---

# 6. Retry Strategy

Many failures are temporary.

Instead of failing immediately:

```text
Call API

↓

Failure

↓

Wait

↓

Retry

↓

Success
```

---

# 7. Exponential Backoff

A production-standard retry strategy.

Instead of

```
Retry immediately
```

Wait longer after each failure.

Example

```
Retry 1

↓

1 second

Retry 2

↓

2 seconds

Retry 3

↓

4 seconds

Retry 4

↓

8 seconds
```

Benefits:

- Reduces server load
- Handles temporary outages
- Prevents retry storms

---

# 8. Retry Flow

```text
API Call

↓

Success?

│

├── Yes

│

▼

Done

│

└── No

↓

Wait

↓

Retry

↓

Success?

↓

Repeat until limit
```

---

# 9. Maximum Retry Limit

Never retry forever.

Example

```python
MAX_RETRIES = 3
```

After 3 failures:

```
Return Error
```

---

# 10. Timeout Handling

Suppose

```
LLM takes

120 seconds
```

Users shouldn't wait forever.

Example

```
Timeout = 30 seconds
```

If exceeded

```
Cancel Request
```

---

# 11. Fallback Models

Production systems often use backup models.

```text
GPT-5

↓

Failure

↓

GPT-4.1

↓

Failure

↓

Smaller Local Model
```

This improves reliability.

---

# 12. Retriever Errors

Problem

```
No documents found
```

Wrong approach

```
Hallucinate
```

Better approach

```
"I couldn't find relevant information in the knowledge base."
```

---

# 13. Output Parsing Errors

Expected

```json
{
"name":"John"
}
```

Received

```
John works in HR.
```

Solution

- Retry
- Ask for structured output
- Validate schema
- Return controlled error

---

# 14. Tool Errors

Suppose

```
Weather API

↓

500 Error
```

Agent options

```
Retry

↓

Use Backup API

↓

Return Error
```

Don't silently ignore failures.

---

# 15. Rate Limiting (429)

Very common interview topic.

When API returns

```
429 Too Many Requests
```

Do **not** immediately retry in a tight loop.

Instead

```
Wait

↓

Retry

↓

Exponential Backoff
```

Respect provider limits.

---

# 16. Circuit Breaker Pattern

Suppose an API is completely down.

Without a circuit breaker

```text
1000 Requests

↓

1000 Failures
```

With a circuit breaker

```text
Several Failures

↓

Circuit Opens

↓

Stop Calling API

↓

Wait

↓

Try Again Later
```

This protects both your application and the external service.

---

# 17. Graceful Degradation

Instead of

```
Application Failed
```

Return

```
Search service is temporarily unavailable.

Please try again in a few minutes.
```

A good user experience matters.

---

# 18. Production Workflow

```text
User

↓

LLM

↓

Failure?

│

├── No

│

▼

Return Answer

│

└── Yes

↓

Retry

↓

Fallback Model

↓

Graceful Error
```

---

# 19. Best Practices

✅ Set request timeouts.

✅ Use exponential backoff.

✅ Limit retries.

✅ Log all failures.

✅ Implement fallback models where appropriate.

✅ Return user-friendly messages.

✅ Monitor failures with LangSmith or other observability tools.

---

# 20. Common Mistakes

❌ Infinite retry loops.

❌ Returning raw exception messages to users.

❌ Ignoring timeout configuration.

❌ Hallucinating when retrieval fails.

❌ Assuming every failure is permanent.

---

# 21. Interview Questions

## Q1. Why is retry needed in LLM applications?

**Answer:**

Many failures, such as temporary network issues or rate limits, are transient. Retrying with an appropriate strategy improves reliability without requiring user intervention.

---

## Q2. What is exponential backoff?

**Answer:**

It is a retry strategy where the waiting time increases after each failed attempt (for example, 1s, 2s, 4s, 8s). This reduces load on external services and improves recovery from temporary failures.

---

## Q3. What should happen if retrieval returns no documents?

**Answer:**

The application should avoid hallucinating. Instead, it should clearly inform the user that no relevant information was found or fall back to another retrieval strategy if available.

---

## Q4. Why should retries have a maximum limit?

**Answer:**

Without a limit, retries can create infinite loops, waste resources, increase costs, and degrade user experience.

---

## Q5. What is a fallback model?

**Answer:**

A fallback model is a secondary LLM that is used if the primary model is unavailable or fails, improving application availability.

---

## Q6. What is graceful degradation?

**Answer:**

Graceful degradation means providing a useful response or reduced functionality instead of crashing when part of the system fails.

---

# 22. Quick Revision

| Concept | Purpose |
|----------|----------|
| Retry | Recover from temporary failures |
| Exponential Backoff | Increase wait time after failures |
| Timeout | Prevent hanging requests |
| Fallback Model | Backup LLM |
| Circuit Breaker | Stop repeated failures |
| Graceful Degradation | Better user experience |

---

# Interview Cheat Sheet

```text
User

↓

LLM

↓

Success?

│

├── Yes

│

▼

Answer

│

└── No

↓

Retry

↓

Exponential Backoff

↓

Fallback Model

↓

Graceful Error

Production Checklist

✓ Timeout

✓ Retry Limit

✓ Logging

✓ Monitoring

✓ Fallback

✓ User-Friendly Errors
```

---

# 23. Real Interview Scenario

**Question:**

> Your production RAG chatbot occasionally fails because the OpenAI API returns `429 Too Many Requests`. How would you handle this?

**Answer:**

I would first identify the error as a rate-limit issue. Then I would implement exponential backoff with a maximum retry count, respect any `Retry-After` headers if provided, and avoid immediate repeated requests. If retries fail, I would either route the request to a fallback model or return a user-friendly message indicating temporary high demand. I would also monitor the frequency of 429 responses to determine whether request throttling, batching, caching, or a higher API quota is needed.

---

# 30-Second Interview Answer

> **Production AI systems interact with multiple external services, so failures are inevitable. A robust error-handling strategy includes exception handling, retries with exponential backoff, request timeouts, fallback models, circuit breakers, and graceful degradation. The goal is to recover from temporary failures when possible and provide clear, reliable responses without exposing internal errors to users.**

---

# Key Takeaway

> **The quality of a production AI system is measured not only by how well it answers questions when everything works, but also by how gracefully it handles failures when things go wrong.**